# 1. Data Pipeline
BOLD -> FFT -> Hilbert pipeline

In [ ]:
from pathlib import Path
import sys
import torch

import matplotlib.pyplot as plt
import numpy as np
#import scienceplots
import matplotlib.patches as patches
#plt.style.use('science')

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.dataset import load_mat_data, fft_bandpass_3d, hilbert_transform
from src.models import CoupledHopfModel
from src.metrics import compute_static_fc, phase_coherence_matrix

def _to_numpy(x: torch.Tensor | np.ndarray) -> np.ndarray:
    if isinstance(x, np.ndarray):
        return x
    return x.detach().cpu().numpy()

plt.rcParams["figure.dpi"] = 140
%matplotlib inline

In [ ]:
# Auto-create parent dirs for savefig (added for reproducible runs)
import matplotlib.figure as _mpl_figure
from pathlib import Path as _Path
_orig_savefig = _mpl_figure.Figure.savefig
def _patched_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, (str, _Path)):
        p = _Path(fname)
        if p.parent and not p.parent.exists():
            p.parent.mkdir(parents=True, exist_ok=True)
    return _orig_savefig(self, fname, *args, **kwargs)
_mpl_figure.Figure.savefig = _patched_savefig


In [ ]:
dataset_path = project_root / "data" / "ts_young" / "ts_young_TR0.72.mat"
subject_index = 0
roi_indices = [0, 49, 99]
focus_roi = 25
dt = 0.72
f_lo, f_hi = 0.008, 0.08
w, h = 2.5, 1.5
v_lim = 2.5
max_timepoints = 200

data = load_mat_data(str(dataset_path))
raw_timeseries = data["timeseries_all"].transpose(2, 0, 1)


print(f"dataset: {dataset_path}")
print(f"subjects x rois x timepoints: {raw_timeseries.shape}")


In [ ]:
ts = torch.as_tensor(raw_timeseries)
time = np.arange(max_timepoints, dtype=np.float32) * dt

subject_ts = ts[subject_index : subject_index + 1]
mean = subject_ts.mean(dim=2, keepdim=True)
std = subject_ts.std(dim=2, keepdim=True) + 1e-8
normalized = (subject_ts - mean) / std

filtered = fft_bandpass_3d(normalized, dt=dt, f_lo=f_lo, f_hi=f_hi)
analytic = hilbert_transform(filtered)

real = _to_numpy(normalized[0, :, :max_timepoints])
filtered = _to_numpy(filtered[0, :, :max_timepoints])
analytic = _to_numpy(analytic[0,: , :max_timepoints])
normalized = _to_numpy(normalized[0, :, :max_timepoints])

In [ ]:
# Colormap for ROIs

cmap = plt.cm.Blues
colors = cmap(np.linspace(0.5, 1, len(roi_indices)))

(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
for i, roi in enumerate(roi_indices):
    fig, ax = plt.subplots(figsize=(w, h))
    fig.patch.set_alpha(0.0)
    # ax.patch.set_alpha(0.8)
    plt.plot(time, normalized[roi], label=f"ROI {roi+1}", alpha=1.0, color=colors[i])
    #ax.set_title(f"Subject {pipeline.subject_index}, ROI {roi}")
    ax.set_yticklabels([]); #ax.set_yticks([-1, 0, 1]);
    ax.set_ylim(-v_lim,v_lim)
    #ax.set_xticks([]); ax.set_xticklabels([])
    plt.legend(loc="upper right", framealpha=1, facecolor="white", frameon=True, edgecolor="white")
    ax.set_xlabel("Time ($s$)")
    plt.savefig(project_root / "paper_new" / "images" / "representation" / f"raw_timeseries_roi{roi}.svg", dpi=100, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"raw_timeseries_roi{roi}.png", dpi=200, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"raw_timeseries_roi{roi}.svg", bbox_inches="tight")

In [ ]:
# Colormap for ROIs

cmap = plt.cm.Blues
colors = cmap(np.linspace(0.5, 1, len(roi_indices)))

(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
for i, roi in enumerate(roi_indices):
    fig, ax = plt.subplots(figsize=(w, h))
    fig.patch.set_alpha(0.0)
    plt.plot(time, filtered[roi], label=f"ROI {roi+1}", alpha=1.0, color=colors[i])
    #ax.set_title(f"Subject {pipeline.subject_index}, ROI {roi}")
    ax.set_yticklabels([]); #ax.set_yticks([])
    ax.set_ylim(-v_lim,v_lim)
    # ax.set_xticks([]); ax.set_xticklabels([])
    ax.set_xlabel("Time ($s$)")
    plt.legend(loc="upper right")
    plt.savefig(project_root / "paper_new" / "images" / "representation" / f"filtered_timeseries_roi{roi}.svg", dpi=100, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"filtered_timeseries_roi{roi}.png", dpi=200, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"filtered_timeseries_roi{roi}.svg", bbox_inches="tight")

In [ ]:

cmap = plt.cm.Blues
colors = cmap(np.linspace(0.5, 1, len(roi_indices)))

(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
for i, roi in enumerate(roi_indices):
    fig, ax = plt.subplots(figsize=(h, h))
    fig.patch.set_alpha(0.0)
    plt.plot(analytic[i].real, analytic[roi].imag, label=f"ROI {roi+1}", alpha=1.0, color=colors[i])
    #ax.set_title(f"Subject {pipeline.subject_index}, ROI {roi}")
    ax.set_yticklabels([]); ax.set_xticklabels([])
    ax.set_xticks([-2, -1, 0, 1, 2]); ax.set_yticks([-2, -1, 0, 1, 2])
    ax.set_ylim(-v_lim, v_lim); ax.set_xlim(-v_lim, v_lim)
    ax.set_xlabel("Real"); ax.set_ylabel("Imaginary")

    ax.set_xticklabels([])
    plt.legend(loc="upper right")
    plt.savefig(project_root / "paper_new" / "images" / "representation" / f"analytic_timeseries_2d_roi{roi}.svg", dpi=100, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"analytic_timeseries_2d_roi{roi}.png", dpi=200, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"analytic_timeseries_2d_roi{roi}.svg", bbox_inches="tight")

In [ ]:
raw_spectrum = np.abs(np.fft.rfft(real[0]))
filtered_spectrum=np.abs(np.fft.rfft(filtered[0]))
frequencies=np.fft.rfftfreq(max_timepoints, d=dt)

fig, ax = plt.subplots(figsize=(h, h))
#fig.patch.set_alpha(0.0)
raw_spec = raw_spectrum / (raw_spectrum.max() + 1e-8)
filtered_spec = filtered_spectrum / (filtered_spectrum.max() + 1e-8)
ax.plot(frequencies, raw_spec, color="#8d99ae", linewidth=2.0, label="BOLD")
ax.plot(
    frequencies,
    filtered_spec,
    color="#d62828",
    linewidth=2.0,
    label="Filtered",
)
ax.axvspan(f_lo, f_hi, color="#ffd166", alpha=0.25, label="Pass band")
ax.set_title(f"[{f_lo:.3f}, {f_hi:.3f}] Hz", fontsize=12, pad=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Relative amplitude")
ax.set_yticks([]); ax.set_xticks([])
ax.set_xlim(left=0.0)
#ax.grid(True, alpha=0.25)
ax.legend(loc="upper right", fontsize=10, frameon=False, bbox_to_anchor=(1.2, 1.0), borderaxespad=0,)
plt.savefig(project_root / "paper_new" / "images" / "representation" / f"Band_pass.svg", dpi=100, bbox_inches="tight")
(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"Band_pass.png", dpi=200, bbox_inches="tight")
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"Band_pass.svg", bbox_inches="tight")

In [ ]:
# Make a 3d plot of the analytic signal 
from mpl_toolkits.mplot3d.axes3d import Axes3D
id = 0
fig = plt.figure(figsize=(int(3*2), int(2)))
fig.patch.set_alpha(0.0)
ax = fig.add_subplot(111, projection='3d')
ax.plot(time, analytic[id].real, analytic[id].imag, linewidth=1.5, alpha=1.0, color="purple", linestyle='solid')
#ax.set_title(f"ROI {roi_indices[id]+1}", y=0.95)
ax.plot(time, v_lim, analytic[id].imag, alpha=0.8, color=colors[id], linestyle='--', label="Imag", zorder=0) # for setting the zlim
ax.plot(time, analytic[id].real, -v_lim, alpha=0.8, color=colors[id], linestyle='-', label="Real") # for setting the zlim
# Remove the grid and leave the horizontal lines only
ax.grid(False)   
ax.set_yticklabels([]); ax.set_zticklabels([])
ax.xaxis.set_tick_params(pad=-3)
ax.set_zticks([-2, -1, 0, 1, 2])
#ax.set_zticklabels(["-2i", "-i", "0", "i", "2i"])
ax.set_yticks([-2, -1, 0, 1, 2])
#ax.set_yticklabels(["-2", "-1", "0", "1", "2"])
ax.set_ylim(-v_lim, v_lim)
ax.set_zlim(-v_lim, v_lim)
ax.set_xlim(time[0]-10, time[-1])
ax.set_xticklabels([])
ax.set_xlabel("Time ($s$)", labelpad=-15)
ax.set_ylabel("Real", labelpad=-12, rotation=40)
ax.set_zlabel("Imaginary", rotation=90, labelpad=-13)
ax.view_init(elev=20, azim=-60)
#ax.set_yticklabels([])
#ax.set_zticklabels([])
# x_scale=1
# y_scale=1
# z_scale=1

# scale=np.diag([x_scale, y_scale, z_scale, 1.0])
# scale=scale*(1.0/scale.max())
# scale[3,3]=1.0

# def short_proj():
#   return np.dot(Axes3D.get_proj(ax), scale)

# ax.get_proj=short_proj

# for z_val in [-2, -1, 0, 1, 2]:
#     ax.plot([time[0], time[-1]], [v_lim, v_lim], [z_val, z_val],
#             color='k', alpha=0.4, linewidth=.5, zorder=0)
#     ax.plot([time[0], time[-1]], [z_val, z_val], [-v_lim,-v_lim],
#             color='k', alpha=0.4, linewidth=.5, zorder=0)
    


#plt.legend(loc="upper right")
plt.savefig(project_root / "paper_new" / "images" / "representation" / f"analytic_signal.svg", dpi=100, bbox_inches="tight")
(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"analytic_signal.png", dpi=200, bbox_inches="tight")
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"analytic_signal.svg", bbox_inches="tight")

# 2. Metrics

In [ ]:
col, row = 33, 68

In [ ]:
cmap = plt.cm.Blues
colors = cmap(np.linspace(0.5, 1, len(roi_indices)))

(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
for i, roi in enumerate([col, row]):
    fig, ax = plt.subplots(figsize=(w, h))
    fig.patch.set_alpha(0.0)
    plt.plot(time, filtered[roi], label=f"Real", color=colors[i])
    ax.text(time[-1]//2, v_lim-0.5, f"ROI {['$i$', '$j$'][i]}", va='center', fontsize=10)
    #ax.set_title(f"Subject {pipeline.subject_index}, ROI {roi}")
    ax.set_yticklabels([]); #ax.set_yticks([])
    ax.set_ylim(-v_lim,v_lim)
    # ax.set_xticks([]); ax.set_xticklabels([])
    ax.set_xlabel("Time ($s$)")
    plt.legend(loc="upper left", framealpha=1, facecolor="white", frameon=True, edgecolor="white")
    plt.savefig(project_root / "paper_new" / "images" / "representation" / f"FC_timeseries_roi{['i','j'][i]}.svg", dpi=100, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"FC_timeseries_roi{['i','j'][i]}.png", dpi=200, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"FC_timeseries_roi{['i','j'][i]}.svg", bbox_inches="tight")

In [ ]:
fc_data = compute_static_fc(torch.tensor(analytic))[0]
fig, ax = plt.subplots(figsize=(h, h))

n = len(fc_data)
mask = np.triu(np.ones((n, n), dtype=bool), k=1)  # Mask upper triangle
masked = np.where(mask, np.nan, fc_data)
ax.imshow(masked, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks([col]); ax.set_xticklabels(['$j$'])
ax.set_yticks([row]); ax.set_yticklabels(['$i$']) 

# Highlight a specific column and row with bounding boxes
box_kw = dict(linewidth=1.5, edgecolor="k", facecolor="none")
ax.set_title("FC matrix")

# Highlight column (only lower triangle cells)
ax.add_patch(patches.Rectangle((0 - 0.5, row - 0.5), col+1, 1, **box_kw))
ax.add_patch(patches.Rectangle((col - 0.5, 100 - 0.5), 1, row-100, **box_kw))

plt.savefig(project_root / "paper_new" / "images" / "representation" / f"FC_matrix.svg", dpi=200, bbox_inches="tight")
(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"FC_matrix.png", dpi=200, bbox_inches="tight")
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"FC_matrix.svg", bbox_inches="tight")

In [ ]:
cmap1 = plt.cm.Blues
colors1 = cmap1(np.linspace(0.5, 1, len(roi_indices)))

(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
for i, roi in enumerate([col, row]):
    fig, ax = plt.subplots(figsize=(w, h))
    fig.patch.set_alpha(0.0)
    ax.plot(time, analytic[roi].real, label=f"Real", alpha=0.8, color=colors1[i])
    ax.plot(time, analytic[roi].imag, label=f"Imag", alpha=0.8, color=colors1[i], linestyle='--')
    ax.text(time[-1]//2-10, v_lim-0.5, f"ROI {['$i$', '$j$'][i]}", va='center', fontsize=10)
    # ax.set_title(f"ROI {['$i$', '$j$'][i]}", y=0.95)
    #ax.set_title(f"Subject {pipeline.subject_index}, ROI {roi}")
    ax.set_yticklabels([]); #ax.set_yticks([])
    ax.set_ylim(-v_lim,v_lim)
    # ax.set_xticks([]); ax.set_xticklabels([])
    ax.set_xlabel("Time ($s$)")
    plt.legend(loc="lower right", framealpha=1, frameon=True, edgecolor="white", ncols=2)
    plt.savefig(project_root / "paper_new" / "images" / "representation" / f"phfc_timeseries_roi{['i','j'][i]}.svg", dpi=100, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"phfc_timeseries_roi{['i','j'][i]}.png", dpi=200, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"phfc_timeseries_roi{['i','j'][i]}.svg", bbox_inches="tight")

In [ ]:
cmap1 = plt.cm.RdPu
colors1 = cmap1(np.linspace(0.5, 1, len(roi_indices)))

fig, ax = plt.subplots(figsize=(w, h))
phases = np.angle(analytic)
diff = np.expand_dims(phases, axis=0) - np.expand_dims(phases, axis=1)
diff = np.cos(diff)
ax.plot(time,diff[col, row], alpha=0.8, label="$\\cos(\\phi_i - \\phi_j)$", color="purple")
ax.set_title("ROI $i-j$")
ax.set_yticklabels([]); #ax.set_yticks([])
# ax.set_xticks([]); ax.set_xticklabels([])
ax.set_xlabel("Time ($s$)")
plt.legend(loc="upper right", )
phFC = diff.mean(axis=-2)
print(phFC[col, row])
# ax.axhline(y=phFC[col, row], color='red', linestyle='--', linewidth=1.5)
# ax.text(x=0.01, y=phFC[col, row] + 0.03, s=f'y = {phFC[col, row]:0.2f}', color='red', transform=ax.get_yaxis_transform())
plt.savefig(project_root / "paper_new" / "images" / "representation" / f"phases_coherence.svg", dpi=100, bbox_inches="tight")
(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"phases_coherence.png", dpi=200, bbox_inches="tight")
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"phases_coherence.svg", bbox_inches="tight")

In [ ]:
fc_data = diff.mean(axis=2)
fig, ax = plt.subplots(figsize=(h, h))

n = len(fc_data)
mask = np.triu(np.ones((n, n), dtype=bool), k=1)  # Mask upper triangle
masked = np.where(mask, np.nan, fc_data)
ax.imshow(masked, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticklabels([])
ax.set_yticklabels([]) 

# Highlight a specific column and row with bounding boxes
box_kw = dict(linewidth=1.5, edgecolor="k", facecolor="none")
ax.set_title("phFC matrix")

plt.savefig(project_root / "paper_new" / "images" / "representation" / f"phFC_matrix_analytic.svg", dpi=200, bbox_inches="tight")
(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"phFC_matrix_analytic.png", dpi=200, bbox_inches="tight")
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"phFC_matrix_analytic.svg", bbox_inches="tight")

# 3. Model

In [ ]:
# Colormap for ROIs

cmap1 = plt.cm.Blues
colors1 = cmap1(np.linspace(0.5, 1, len(roi_indices)))

(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
for i, roi in enumerate(roi_indices):
    fig, ax = plt.subplots(figsize=(w, h))
    fig.patch.set_alpha(0.0)
    ax.plot(time, analytic[roi].real, label=f"Real", alpha=0.8, color=colors1[i])
    ax.plot(time, analytic[roi].imag, label=f"Imag", alpha=0.8, color=colors1[i], linestyle='--')
    ax.text(time[-1]//2 -10, v_lim-0.5, f"ROI {roi+1}", va='center', fontsize=10)
    # ax.set_title(f"ROI {roi+1}", y=0.97)
    #ax.set_title(f"Subject {pipeline.subject_index}, ROI {roi}")
    ax.set_yticklabels([]); #ax.set_yticks([])
    ax.set_ylim(-v_lim,v_lim)
    # ax.set_xticks([]); ax.set_xticklabels([])
    ax.set_xlabel("Time ($s$)")
    plt.legend(loc="lower right", framealpha=1, frameon=True, edgecolor="white", ncols=2)
    plt.savefig(project_root / "paper_new" / "images" / "representation" / f"analytic_timeseries_roi{roi}.svg", dpi=100, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"analytic_timeseries_roi{roi}.png", dpi=200, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"analytic_timeseries_roi{roi}.svg", bbox_inches="tight")

In [ ]:
model = CoupledHopfModel(
    n_rois=100,
    initial_a=-0.02,   # Bifurcation parameter (near criticality)
    initial_g=1.0,      # Global coupling strength
    initial_kappa=0.1,  # Scaling factor for local dynamics
    noise_sigma=0.1,    # Noise amplitude (default)
)
with torch.no_grad():
    timeseries = model.forward(initial_state=torch.tensor(analytic[None, :, 0]), n_steps=max_timepoints)  # (10, 68, 300) complex
    fc_matrix = model.compute_fc(timeseries)
timeseries = timeseries.detach().numpy()

In [ ]:
# Colormap for ROIs
cmap1 = plt.cm.Blues
colors1 = cmap1(np.linspace(0.5, 1, len(roi_indices)))
cmap2 = plt.cm.Reds
colors2 = cmap2(np.linspace(0.5, 1, len(roi_indices)))

(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
for i, roi in enumerate(roi_indices):
    fig, ax = plt.subplots(figsize=(w, h))
    fig.patch.set_alpha(0.0)
    ax.plot(time, timeseries[0, roi].real, label=f"Real", alpha=0.8, color=colors1[i])
    ax.plot(time, timeseries[0, roi].imag, label=f"Imag", alpha=0.8, color=colors1[i], linestyle='--')
    ax.text(time[-1]//2 -10, v_lim-0.5, f"ROI {roi+1}", va='center', fontsize=10)
    # ax.set_title(f"ROI {roi+1}", y=0.97)
    #ax.set_title(f"Subject {pipeline.subject_index}, ROI {roi}")
    ax.set_yticklabels([]); #ax.set_yticks([])
    ax.set_ylim(-v_lim,v_lim)
    # ax.set_xticks([]); ax.set_xticklabels([])
    ax.set_xlabel("Time ($s$)")
    plt.legend(loc="lower right", framealpha=1, frameon=True, edgecolor="white", ncols=2)
    plt.savefig(project_root / "paper_new" / "images" / "representation" / f"model_timeseries_roi{roi}.svg", dpi=100, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"model_timeseries_roi{roi}.png", dpi=200, bbox_inches="tight")
    plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"model_timeseries_roi{roi}.svg", bbox_inches="tight")

In [ ]:
fc_data = compute_static_fc(torch.tensor(analytic))[0]
fig, ax = plt.subplots(figsize=(w, w))
fig.patch.set_alpha(0.0)

n = len(fc_data)
mask = np.triu(np.ones((n, n), dtype=bool), k=1)  # Mask upper triangle
masked = np.where(mask, np.nan, fc_data)
ax.imshow(masked, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticklabels([])
ax.set_yticklabels([]) 

# Highlight a specific column and row with bounding boxes
box_kw = dict(linewidth=1.5, edgecolor="k", facecolor="none")
ax.set_title("FC matrix")


plt.savefig(project_root / "paper_new" / "images" / "representation" / f"FC_matrix_analytic.svg", dpi=100, bbox_inches="tight")
(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"FC_matrix_analytic.png", dpi=200, bbox_inches="tight")
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"FC_matrix_analytic.svg", bbox_inches="tight")

In [ ]:
fc_data = compute_static_fc(torch.tensor(timeseries))[0]
fig, ax = plt.subplots(figsize=(w, w))
fig.patch.set_alpha(0.0)

n = len(fc_data)
mask = np.triu(np.ones((n, n), dtype=bool), k=1)  # Mask upper triangle
masked = np.where(mask, np.nan, fc_data)
ax.imshow(masked, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticklabels([])
ax.set_yticklabels([]) 

# Highlight a specific column and row with bounding boxes
box_kw = dict(linewidth=1.5, edgecolor="k", facecolor="none")
ax.set_title("FC matrix")

plt.savefig(project_root / "paper_new" / "images" / "representation" / f"FC_matrix_model.svg", dpi=100, bbox_inches="tight")
(project_root / "paper_new" / "images_png" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_png" / "representation" / f"FC_matrix_model.png", dpi=200, bbox_inches="tight")
(project_root / "paper_new" / "images_svg" / "representation").mkdir(parents=True, exist_ok=True)
plt.savefig(project_root / "paper_new" / "images_svg" / "representation" / f"FC_matrix_model.svg", bbox_inches="tight")

# 4. Stacked pipeline signals
Three ROIs stacked vertically per pipeline step (raw → processed → analytic), plus the Hopf-model output. Saves to `paper_new/images_*/representation/`.

In [ ]:
out_dir_pdf = project_root / "paper_new" / "images" / "representation"
out_dir_png = project_root / "paper_new" / "images_png" / "representation"
out_dir_svg = project_root / "paper_new" / "images_svg" / "representation"
for d in (out_dir_pdf, out_dir_png, out_dir_svg):
    d.mkdir(parents=True, exist_ok=True)


def stacked_plot(signals, filename, signals_imag=None, colors="blues"):
    """Vertically stacked ROI signals for one pipeline step.

    signals       : (n_rois_selected, n_time) real-valued array
    signals_imag  : optional (same shape), drawn as dashed lines on top
    colors        : 'blues' (default) or 'reds'
    """
    cmap = plt.cm.Blues if colors == "blues" else plt.cm.Reds
    line_colors = cmap(np.linspace(0.5, 1.0, len(roi_indices)))

    amps = [np.max(np.abs(s)) for s in signals]
    if signals_imag is not None:
        amps += [np.max(np.abs(s)) for s in signals_imag]

    spacing = 2.2 * max(amps)
    offsets = [spacing * (len(signals) - 1 - i) for i in range(len(signals))]
    # Gap below the last ROI to host the vertical-dots separator.
    offsets[-1] -= spacing * 0.7

    fig, ax = plt.subplots(figsize=(w, h))
    fig.patch.set_alpha(0.0)

    for i, s in enumerate(signals):
        ax.plot(time, s + offsets[i], color=line_colors[i], linewidth=1.2, alpha=1.0)
        if signals_imag is not None:
            ax.plot(time, signals_imag[i] + offsets[i],
                    color=line_colors[i], linewidth=1.2, alpha=1.0, linestyle="--")

    if len(signals) >= 2:
        mid_x = (time[0] + time[-1]) / 2.0
        dot_ys = np.linspace(offsets[-1] + spacing * 0.65,
                             offsets[-2] - spacing * 0.65, 3)
        ax.plot([mid_x] * 3, dot_ys, 'o', color='k', markersize=1, zorder=3)

    y_min = -spacing * 0.6 - spacing * 0.7
    y_max = offsets[0] + spacing * 0.6
    ax.set_ylim(y_min, y_max)
    ax.set_xlim(time[0], time[-1])

    ax.set_xticks([]); ax.set_yticks([])
    for side in ("top", "right", "bottom", "left"):
        ax.spines[side].set_visible(False)

    x_span = time[-1] - time[0]
    ax.plot([time[0], time[0]], [y_min, y_max],
            color="black", lw=1.0, solid_capstyle="butt", clip_on=False)
    ax.annotate("", xy=(time[-1] + 0.04 * x_span, y_min), xytext=(time[0], y_min),
                arrowprops=dict(arrowstyle="->", color="black", lw=1.0,
                                shrinkA=0, shrinkB=0),
                annotation_clip=False)
    ax.set_xlabel("Time", labelpad=8)

    fig.savefig(out_dir_pdf / f"{filename}.svg", dpi=200, bbox_inches="tight")
    fig.savefig(out_dir_png / f"{filename}.png", dpi=200, bbox_inches="tight")
    fig.savefig(out_dir_svg / f"{filename}.svg", bbox_inches="tight")
    plt.show()
    return fig


## Step 1 — raw

In [ ]:
raw_signals = np.stack([normalized[roi] for roi in roi_indices])
stacked_plot(raw_signals, "stacked_raw")


## Step 2 — processed (bandpass)

In [ ]:
processed_signals = np.stack([filtered[roi] for roi in roi_indices])
stacked_plot(processed_signals, "stacked_processed")


## Step 3 — analytic signal

In [ ]:
analytic_real = np.stack([analytic[roi].real for roi in roi_indices])
analytic_imag = np.stack([analytic[roi].imag for roi in roi_indices])
stacked_plot(analytic_real, "stacked_analytic", signals_imag=analytic_imag)


## Step 4 — Hopf model output

In [ ]:
torch.manual_seed(0)
_hopf = CoupledHopfModel(
    n_rois=analytic.shape[0],
    initial_a=-0.02,
    initial_g=1.0,
    initial_kappa=0.1,
    noise_sigma=0.1,
)
_ic = torch.tensor(analytic[:, 0])[None]
with torch.no_grad():
    _sim = _hopf.forward(initial_state=_ic, n_steps=max_timepoints)
_sim = _to_numpy(_sim[0])
sim_real = np.stack([_sim[roi].real for roi in roi_indices])
sim_imag = np.stack([_sim[roi].imag for roi in roi_indices])
stacked_plot(sim_real, "stacked_hopf", signals_imag=sim_imag, colors="reds")


# 5. Stacked model trajectories
One plot per ROI: empirical trajectory on top, each model's *N* stochastic simulations stacked below with mean ± 1 std band. Saves to `paper_new/images_*/ts_young/`.

In [ ]:
from src.dataset import load_dataset
from src.models import load_model_from_checkpoint
from src.training import HopfConfig

DATASET_TYPE   = 'ts_young'
DATA_PATH      = 'data/ts_young/ts_young_TR0.72.mat'
CHECKPOINT_DIR = 'checkpoints'
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

STACKED_N_TIMEPOINTS  = 200
STACKED_N_SIMULATIONS = 3
STACKED_ROI_INDICES   = [0]
STACKED_SUBJECT_INDEX = 0

stacked_out_pdf = project_root / 'paper_new' / 'images' / DATASET_TYPE
stacked_out_png = project_root / 'paper_new' / 'images_png' / DATASET_TYPE
stacked_out_svg = project_root / 'paper_new' / 'images_svg' / DATASET_TYPE
for d in (stacked_out_pdf, stacked_out_png, stacked_out_svg):
    d.mkdir(parents=True, exist_ok=True)

_cfg = HopfConfig()
_cfg.dataset_type = DATASET_TYPE
_cfg.data_path    = DATA_PATH
_cfg.use_wandb    = False
stacked_dataset = load_dataset(_cfg, DEVICE)
print(f'n_rois={stacked_dataset.n_rois}  '
      f'n_subjects={stacked_dataset.timeseries.shape[0]}  '
      f'n_timepoints={stacked_dataset.n_timepoints}  dt={stacked_dataset.dt}')


In [ ]:
_MODEL_TAGS = [
    ('hybrid_neural', 'Hybrid+Neural'),
    ('hybrid_hopf',   'Hybrid Hopf'),
    ('gnn_hopf',      'GNN Hopf'),
    ('nsde',          'Neural SDE'),
    ('hopf',          'Hopf'),
]

def _short_label(stem: str) -> str:
    lower = stem.lower()
    for tag, label in _MODEL_TAGS:
        if tag in lower:
            return label + (' (Grid)' if 'grid' in lower else '')
    return stem

ckpt_dir = Path(CHECKPOINT_DIR)
checkpoint_paths = sorted(ckpt_dir.glob(f'*{DATASET_TYPE}*.pt'))

stacked_models = {}
for ckpt_path in checkpoint_paths:
    try:
        m, _, _ = load_model_from_checkpoint(str(ckpt_path), device=DEVICE)
        if m.n_rois != stacked_dataset.n_rois:
            continue
        stacked_models[_short_label(ckpt_path.stem)] = m
    except Exception as exc:
        print(f'  Failed to load {ckpt_path.name}: {exc}')
print(f'Loaded {len(stacked_models)} model(s): {list(stacked_models.keys())}')


In [ ]:
real_ts = stacked_dataset.timeseries[STACKED_SUBJECT_INDEX]
real_plot = (real_ts.real if torch.is_complex(real_ts) else real_ts)[:, :STACKED_N_TIMEPOINTS]
real_plot = real_plot.detach().cpu().numpy()

ic_repeated = real_ts[:, 0].unsqueeze(0).expand(STACKED_N_SIMULATIONS, -1)

torch.manual_seed(0)
sim_per_model = {}
for name, model in stacked_models.items():
    with torch.no_grad():
        sim_ts = model.forward(initial_state=ic_repeated, n_steps=STACKED_N_TIMEPOINTS)
    sim_real_arr = sim_ts.real if torch.is_complex(sim_ts) else sim_ts
    sim_per_model[name] = sim_real_arr.detach().cpu().numpy()
    print(f'  {name}: simulated shape={sim_per_model[name].shape}')


In [ ]:
def stacked_trajectories(
    real_signal,
    sims_per_model,
    models_to_plot=("Hopf", "Neural SDE", "Hybrid Hopf"),
    *,
    dt=1.0,
    figsize=(2.5, 0.7),
    save_stem=None,
):
    if models_to_plot is not None:
        sims_per_model = {n: sims_per_model[n] for n in models_to_plot if n in sims_per_model}

    n_rows = 1 + len(sims_per_model)
    T = real_signal.shape[0]
    t = np.arange(T) * dt

    fig, axes = plt.subplots(n_rows, 1,
                             figsize=(figsize[0], figsize[1] * n_rows),
                             sharex=True, gridspec_kw={'hspace': 0.3})
    fig.patch.set_alpha(0.0)
    if n_rows == 1:
        axes = [axes]

    axes[0].plot(t, real_signal, color='#1f77b4', lw=1.4, label='Empirical')
    axes[0].legend(loc='upper right', frameon=False, fontsize=9,
                   bbox_to_anchor=(1.0, 1.3))

    red_colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(sims_per_model)))
    for i, (name, sims) in enumerate(sims_per_model.items(), start=1):
        ax = axes[i]
        color = red_colors[i - 1]
        mu = sims.mean(axis=0)
        sd = sims.std(axis=0)
        for k in range(sims.shape[0]):
            ax.plot(t, sims[k], color=color, lw=0.5, alpha=0.35)
        ax.fill_between(t, mu - sd, mu + sd, color=color, alpha=0.20, linewidth=0)
        ax.plot(t, mu, color=color, lw=1.4, alpha=0.95, label=name)
        ax.legend(loc='upper right', frameon=False, fontsize=9,
                  bbox_to_anchor=(1.0, 1.5))

    for i, ax in enumerate(axes):
        for side in ('top', 'right', 'left'):
            ax.spines[side].set_visible(False)
        ax.set_yticks([])
        if i < len(axes) - 1:
            ax.spines['bottom'].set_visible(False)
            ax.tick_params(bottom=False, labelbottom=False)
        else:
            ax.tick_params(bottom=False, labelbottom=True)
        ax.set_xlim(t[0], t[-1])

    axes[-1].set_xlabel('Time (s)')
    axes[-1].set_xticks([])
    axes[-1].plot(t[-1], 0, marker='>', color='black', markersize=6,
                  transform=axes[-1].get_xaxis_transform(), clip_on=False)
    fig.subplots_adjust(hspace=0.0)

    if save_stem is not None:
        fig.savefig(stacked_out_pdf / f'{save_stem}.pdf', bbox_inches='tight')
        fig.savefig(stacked_out_png / f'{save_stem}.png', bbox_inches='tight', dpi=200)
        fig.savefig(stacked_out_svg / f'{save_stem}.svg', bbox_inches='tight')
    return fig


for roi in STACKED_ROI_INDICES:
    sims_for_roi = {name: sims[:, roi, :] for name, sims in sim_per_model.items()}
    fig = stacked_trajectories(
        real_signal=real_plot[roi],
        sims_per_model=sims_for_roi,
        dt=stacked_dataset.dt,
        save_stem=f'stacked_model_trajectories_roi{roi:03d}',
    )
    plt.show()
    plt.close(fig)
